In [0]:
%run ./00_imports

In [0]:

# ============================================================================
# On vérifie les types, colonnes et valeurs avant toute transformation
# ============================================================================

import json
from pyspark.sql import functions as F

# Chargement de la configuration
with open("/Workspace/Users/sabineanoko@gmail.com/pipeline-qualite-eau/config/pipeline_config.json", "r") as f:
    config = json.load(f)

TABLES = config["tables"]

# Lecture des 3 tables Bronze
df_result = spark.table(TABLES["bronze_result"])
df_plv    = spark.table(TABLES["bronze_plv"])
df_udi    = spark.table(TABLES["bronze_udi"])

# Schéma de la table RESULT
print("Schéma RESULT :")
df_result.printSchema()

# Aperçu des premières lignes
print("\nAperçu RESULT (3 lignes) :")
df_result.limit(3).display(truncate=50)

In [0]:

# =========================================================================================================================================
# Exploration des valeurs distinctes des colonnes clés car qualitparam : N ou O —> Probablement la conformité (modalités à identifier)
# On identifie les modalités avant de définir les règles de transformation
# =========================================================================================================================================

print("Valeurs distinctes de qualitparam (conformité) :")
df_result.groupBy("qualitparam").count().orderBy("count", ascending=False).show()

print("Valeurs distinctes de insituana :")
df_result.groupBy("insituana").count().orderBy("count", ascending=False).show()

print("Valeurs distinctes de rqana (5 premières) :")
df_result.groupBy("rqana").count().orderBy("count", ascending=False).limit(10).display()

### Synthèse intermédiaire :

#### `qualitparam` est la colonne centrale du projet 

C'est elle qui dit si un paramètre analysé est conforme ou non. Les valeurs probables sont :
N = Non-conforme — le paramètre dépasse le seuil réglementaire
O = Conforme (Oui) — le paramètre est dans les normes
S = peut-être "Sans objet" pour les paramètres qualitatifs

A ce stade, on a identifié :

- **N** : 35M lignes  
  N est majoritaire — donc N signifie probablement **"Normal" (conforme)** du moins l'hypothèse à ce stade.  
  C'est contre-intuitif mais logique : 92% de conformité sur l'eau potable française est cohérent avec la réalité (pourcentage d'analyses non-conformes, pas de communes).

- **O** : 3M lignes  
  O = **"Outsider" (non-conforme)** — seulement 8% des analyses.

---
#### `insituana` indique où l'analyse a été faite :

- L = Laboratoire — analyse différée
- T = Terrain — analyse sur place (pH, température, chlore...)

---
#### `rqana` est le résultat brut de l'analyse 

C'est une valeur texte libre : <10, Aspect normal, ... C'est ce qu'a mesuré le technicien.

- Beaucoup de valeurs `<0,005`, `<1`, `<0,020` — sont des valeurs sous le seuil de détection, donc conformes.
- Aucun changement — paramètre qualitatif stable.
- Aspect normal — analyse visuelle conforme.

---
#### La relation entre les trois :

| Colonne         | Exemple de valeur     | Signification                  |
|-----------------|----------------------|-------------------------------|
| `rqana`         | `<10 µg/L`           | Valeur mesurée                |
| `limitequal`    | `<=200 µg/L`         | Seuil réglementaire           |
| `qualitparam`   | `N`                  | Conclusion : Non-conforme ?<br>(attention : N = conforme ici ?) |

C'est justement ce qu'on va vérifier avec la cellule suivante — 

---

**Conclusion pour la couche Silver :**  
On regarde les vraies valeurs pour confirmer ce que signifie N et O et la colonne de conformité qu'on créera sera probablement :

- `qualitparam = "N"` → `conforme = True`
- `qualitparam = "O"` → `conforme = False`

In [0]:

# ============================================================================
# Nettoyage, typage et enrichissement de RESULT
# Standardisation des colonnes, cast des types, création de la colonne conforme
# Pour les paramètres qualitatifs, O signifie "Oui conforme" et non "Outsider"
# ============================================================================

df_silver = df_result \
    .withColumnRenamed("cddept", "code_departement") \
    .withColumnRenamed("referenceprel", "reference_prelevement") \
    .withColumnRenamed("cdparametre", "code_parametre") \
    .withColumnRenamed("libmajparametre", "libelle_parametre") \
    .withColumnRenamed("qualitparam", "code_qualite") \
    .withColumnRenamed("rqana", "resultat_analyse") \
    .withColumnRenamed("valtraduite", "valeur_numerique") \
    .withColumnRenamed("limitequal", "limite_qualite") \
    .withColumnRenamed("cdunitereference", "unite_reference") \
    .withColumnRenamed("insituana", "lieu_analyse") \
    .drop("cdparametresiseeaux", "cdunitereferencesiseeaux", 
          "libminparametre", "libwebparametre", "refqual", "casparam") \
    .withColumn("conforme", 
        F.when(
            F.col("libelle_parametre").rlike("QUALITATIF|O/N|PRES/ABS|CALCOCARBONIQUE"),
            F.when(F.col("code_qualite").isin("N", "O"), True).otherwise(None)
        ).otherwise(
            F.when(F.col("code_qualite") == "N", True)
             .when(F.col("code_qualite") == "O", False)
             .otherwise(None)
        )
    ) \
    .withColumn("valeur_numerique", 
        F.col("valeur_numerique").cast("double")
    ) \
    .withColumn("code_departement",
        F.lpad(F.col("code_departement"), 3, "0")
    ) \
    .withColumn("_source_layer", F.lit("silver"))

print("Schéma Silver :")
df_silver.printSchema()

print(f"\nNombre de lignes : {df_silver.count()}")
print(f"\nRépartition conformité :")
df_silver.groupBy("conforme").count().display()

In [0]:

# ============================================================================
#  Jointure avec PLV pour ajouter dates et communes
# On récupère la date de prélèvement et le code commune depuis la table PLV
# ============================================================================

# Schéma PLV d'abord
print("Colonnes PLV disponibles :")
df_plv.columns

### Colonnes utiles pour la couche Silver

- **referenceprel** : clé de jointure avec RESULT  
- **dateprel** : date du prélèvement  
- **inseecommuneprinc** : code INSEE commune  
- **nomcommuneprinc** : nom de la commune  
- **conclusionprel** : conclusion globale du prélèvement  
- **plvconformitebacterio** : conformité bactériologique  
- **plvconformitechimique** : conformité chimique

In [0]:

# ============================================================================
# Jointure Silver — RESULT enrichi avec PLV
# Déduplication de PLV avant jointure pour éviter l'explosion de lignes
# ============================================================================

from pyspark.sql.window import Window

# Déduplication PLV — on garde la première ligne par referenceprel + annee
fenetre = Window.partitionBy("referenceprel", "_annee").orderBy("dateprel")

df_plv_dedup = df_plv.withColumn("rang", F.row_number().over(fenetre)) \
    .filter(F.col("rang") == 1) \
    .drop("rang") \
    .select(
        F.col("referenceprel").alias("reference_prelevement"),  # Cohérence avec Silver
        F.col("dateprel"),
        F.col("inseecommuneprinc").alias("code_commune"),
        F.col("nomcommuneprinc").alias("nom_commune"),
        "conclusionprel",
        "plvconformitebacterio",
        "plvconformitechimique",
        "_annee"
    ) \
    .withColumn("date_prelevement", 
        F.expr("try_cast(dateprel as date)")
    ) \
    .drop("dateprel")

# Jointure LEFT avec PLV dédupliqué
df_silver_enrichi = df_silver.join(
    df_plv_dedup,
    on=["reference_prelevement", "_annee"],  # Nom standardisé
    how="left"
) \
.withColumn("annee_prelevement", F.year(F.col("date_prelevement")))

# Vérification
nb_avant = df_silver.count()
nb_apres = df_silver_enrichi.count()

print(f"Lignes avant jointure : {nb_avant}")
print(f"Lignes après jointure : {nb_apres}")
print(f"Différence            : {nb_apres - nb_avant}")

display(df_silver_enrichi.select(
    "code_departement", "code_commune", "nom_commune",
    "date_prelevement", "libelle_parametre", "conforme"
).limit(5))

In [0]:

# ============================================================================
# Partitionnement par année et département pour optimiser les lectures Gold
# ============================================================================

df_silver_enrichi.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"_annee IN (2023, 2024, 2025)") \
    .partitionBy("_annee", "code_departement") \
    .saveAsTable(TABLES["silver"])

nb_silver = spark.table(TABLES["silver"]).count()
print(f"Table Silver écrite : {TABLES['silver']}")
print(f"Lignes totales      : {nb_silver}")

display(spark.table(TABLES["silver"]).limit(5))